### 1) Imports y carga

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
df = pd.read_csv("../data/adaptive_comfort_dataset.csv")

print("Total filas:", len(df))
df["cooling_type"].value_counts()

Total filas: 300


cooling_type
air conditioned         182
naturally ventilated     87
mixed mode               31
Name: count, dtype: int64

### 2) Función para experimento sin one-hot

In [5]:
def run_single_cooling_experiment(df_subset, nombre):
    """
    Entrena modelo usando SOLO:
    - t_out_mean
    Predice:
    - neutral_temp
    """

    if len(df_subset) < 10:
        print(f"\n{nombre}: Muy pocos datos ({len(df_subset)})")
        return None

    X = df_subset[["t_out_mean"]].values.astype(np.float32)
    y = df_subset["neutral_temp"].values.astype(np.float32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(n_estimators=15, max_depth=4, random_state=42)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n{nombre}")
    print("Filas:", len(df_subset))
    print("RMSE:", rmse)
    print("R2:", r2)

    return rmse, r2, len(df_subset)

### 3) Experimentos:
- Air Conditioned
- Mixed Mode
- Naturally Ventilated

In [6]:
df_air = df[df["cooling_type"] == "air conditioned"].copy()

res_air = run_single_cooling_experiment(df_air, "AIR CONDITIONED")


AIR CONDITIONED
Filas: 182
RMSE: 1.7128847215381566
R2: -0.14788212895030406


In [7]:
df_mixed = df[df["cooling_type"] == "mixed mode"].copy()

res_mixed = run_single_cooling_experiment(df_mixed, "MIXED MODE")


MIXED MODE
Filas: 31
RMSE: 1.1988423098039163
R2: 0.41187059321075736


In [8]:
df_nat = df[df["cooling_type"] == "naturally ventilated"].copy()

res_nat = run_single_cooling_experiment(df_nat, "NATURALLY VENTILATED")


NATURALLY VENTILATED
Filas: 87
RMSE: 3.3847015888893712
R2: 0.24035194093901324


In [9]:
resultados = pd.DataFrame(
    {
        "Tipo": ["Air Conditioned", "Mixed Mode", "Naturally Ventilated"],
        "Filas": [
            res_air[2] if res_air else 0,
            res_mixed[2] if res_mixed else 0,
            res_nat[2] if res_nat else 0,
        ],
        "RMSE": [
            res_air[0] if res_air else None,
            res_mixed[0] if res_mixed else None,
            res_nat[0] if res_nat else None,
        ],
        "R2": [
            res_air[1] if res_air else None,
            res_mixed[1] if res_mixed else None,
            res_nat[1] if res_nat else None,
        ],
    }
)

resultados

,Tipo,Filas,RMSE,R2
0,Air Conditioned,182,1.712885,-0.147882
1,Mixed Mode,31,1.198842,0.411871
2,Naturally Ventilated,87,3.384702,0.240352
